# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LaibaTaseen/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1: What Makes Content Grow?

The paper found that growing pages are about **37.6% longer** and **20% younger** on average than declining pages. This was based on comparing 74,800 growing pages with 45,600 declining pages.

**My question:** When was the word count measured? Was it when the page was first published, or at the time of the study? If it was measured later, there could be a problem because pages might become longer *because* they are already performing well. So, the data shows a connection between longer pages and growth, but it doesn't necessarily prove that making a page longer will cause it to grow. I think the paper is fair about this, but the suggestion to "expand thin pages" could sound more certain than the evidence actually is.

## Finding 2: Clicks and Search Position

The paper found that CTR drops a lot as a page's search position gets worse — from **0.423% in the top 3 positions** to only **0.050% in deeper positions**, which is an 88% drop.

**My question:** How many pages were included in each position group? Since the CTR is based on total clicks and impressions, a few very high-traffic pages could affect the average. It would be helpful to know the number of pages in each group to see whether this pattern applies broadly or is mainly driven by a few large pages.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [4]:
model_data = con.sql(f"""
    WITH march_data AS (
        SELECT * FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    ),
    bounds AS (SELECT MAX(report_date) AS end_d FROM march_data),
    windowed AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_recent,
               SUM(CASE WHEN report_date <= b.end_d - INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_earlier,
               STDDEV(gsc_avg_position) AS pos_volatility,
               AVG(gsc_avg_position) AS avg_position
        FROM march_data f, bounds b
        GROUP BY 1, 2
        HAVING imp_earlier >= 50
    )
    SELECT * FROM windowed
""").df()

model_data['pct_change'] = (model_data['imp_recent'] - model_data['imp_earlier']) / model_data['imp_earlier']

def label_status(row):
    if row['pct_change'] >= 0.20:
        return 'growing'
    elif row['pct_change'] <= -0.20:
        return 'declining'
    else:
        return 'worth_review'

model_data['status'] = model_data.apply(label_status, axis=1)
print(model_data['status'].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

status
declining       35343
growing         29688
worth_review    29528
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

feature_cols = ['imp_earlier', 'pos_volatility', 'avg_position']
X = model_data[feature_cols]
y = model_data['status']
groups = model_data['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
model_accuracy = accuracy_score(y_te, model.predict(X_te))

print(f"Model accuracy: {model_accuracy:.3f}")
print(classification_report(y_te, model.predict(X_te), digits=3))

Model accuracy: 0.421
              precision    recall  f1-score   support

   declining      0.547     0.473     0.507     11296
     growing      0.368     0.392     0.380      7386
worth_review      0.316     0.364     0.338      6770

    accuracy                          0.421     25452
   macro avg      0.410     0.410     0.408     25452
weighted avg      0.433     0.421     0.425     25452



In [6]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']

model_data_full = model_data.merge(qsignals, on='content_hash_id', how='left').dropna(
    subset=['visible_queries', 'rare_share', 'anon_share', 'top_query_share']
)
print(f'{len(model_data_full):,} rows after join')

feature_cols2 = ['imp_earlier', 'pos_volatility', 'avg_position',
                  'visible_queries', 'rare_share', 'anon_share', 'top_query_share']

X2 = model_data_full[feature_cols2]
y2 = model_data_full['status']
groups2 = model_data_full['client_hash_id']

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx2, test_idx2 = next(gss2.split(X2, y2, groups=groups2))

X_tr2, X_te2 = X2.iloc[train_idx2], X2.iloc[test_idx2]
y_tr2, y_te2 = y2.iloc[train_idx2], y2.iloc[test_idx2]

model2 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr2, y_tr2)
model2_accuracy = accuracy_score(y_te2, model2.predict(X_te2))
baseline2_accuracy = y_te2.value_counts(normalize=True).max()

print(f"Baseline accuracy: {baseline2_accuracy:.3f}")
print(f"Model accuracy: {model2_accuracy:.3f}")
print(f"Improvement: {model2_accuracy - baseline2_accuracy:.3f}")
print(classification_report(y_te2, model2.predict(X_te2), digits=3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

79,800 rows after join
Baseline accuracy: 0.349
Model accuracy: 0.471
Improvement: 0.123
              precision    recall  f1-score   support

   declining      0.531     0.486     0.508     10978
     growing      0.455     0.585     0.512     10036
worth_review      0.429     0.355     0.389     11245

    accuracy                          0.471     32259
   macro avg      0.472     0.475     0.469     32259
weighted avg      0.472     0.471     0.467     32259



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split

# BEFORE: plain random split (not grouped) — pages from the same client can leak across train/test
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X2, y2, test_size=0.25, random_state=42, stratify=y2
)
model_random = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_r, y_tr_r)
random_accuracy = accuracy_score(y_te_r, model_random.predict(X_te_r))

print(f"BEFORE — random split accuracy: {random_accuracy:.3f}")

# AFTER: grouped split (already built as model2 in Week 5) — re-shown here for direct comparison
print(f"AFTER — grouped split accuracy: {model2_accuracy:.3f}")

print(f"\nDifference (random − grouped): {random_accuracy - model2_accuracy:.3f}")


BEFORE — random split accuracy: 0.522
AFTER — grouped split accuracy: 0.471

Difference (random − grouped): 0.050


Before/after split comparison: Using a plain random split gave 0.522 accuracy, while the grouped split (test clients unseen during training) gave 0.471 — a 5-point drop. This confirms that some of the original accuracy came from the model partially recognizing client-specific patterns rather than fully general trends. The grouped-split number (0.471) is the more honest, trustworthy result, and is the one that should be reported and trusted going forward.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage Audit — Checking Each Feature

* **imp_earlier:** Safe. It only uses data from the earlier part of March, before the period used to create the label.
* **pos_volatility:** Small risk. It was calculated using the whole March period, so it includes some data from the same period used for the label.
* **avg_position:** Same small risk as `pos_volatility` because it also uses the full March period.
* **visible_queries:** Safe. It comes from a separate 90-day query summary and doesn't overlap with the March split.
* **rare_share:** Safe for the same reason — it comes from the query data, not the daily March data.
* **anon_share:** Safe, using the same separate query-level data.
* **top_query_share:** Safe for the same reason.

**Overall:** Most of the features are safe, but `pos_volatility` and `avg_position` have a small leakage risk because they use the full month instead of only the earlier period. It's not a major leak, but to make the model more reliable, I would calculate these two features using only the data available before the decision point.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

##My boldest claim, rewritten:
Original: "Start with high-traffic pages marked 'Refresh Soon,' since updating these is likely to have the biggest impact."
Problem: This implies a causal effect, that updating causes improvement, which the model never tested. It only observed patterns associated with decline, not the outcome of an actual fix.
Rewritten: "High-traffic pages marked 'Refresh Soon' are where the model observed the strongest decline signal combined with meaningful traffic, a reasonable starting point for review, though the model does not measure the effect of an actual update."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.